In [1]:
import json

def format_prompts(example):
    """
    Takes your specific JSONL row, isolates the target event, 
    and formats the rest as the input string.
    """
    # 1. Extract the ground-truth label
    target_event = example["event"]
    
    # 2. Build the input dictionary (Excluding 'event' and 'scene_id')
    input_dict = {
        "environment": example["environment"],
        "nodes": example["nodes"],
        "edges": example["edges"]
    }
    
    # 3. Convert the stripped dictionary back into a JSON string
    input_json_str = json.dumps(input_dict)
    
    # 4. Define the strict system instruction
    instruction = (
        "Analyze the scene graph JSON and classify the safety event. "
        "The classes are: 'Safe', 'Gun Detection', 'Knife Detection', 'Fall Detection', 'Fire Detection'."
    )
    
    # 5. Assemble the final training text block
    full_prompt = f"Instruction: {instruction}\nInput: {input_json_str}\nOutput: {target_event}"
    
    return {"text": full_prompt}

# Apply this to your loaded dataset:
# dataset = load_dataset("json", data_files="your_data.jsonl", split="train")
# dataset = dataset.map(format_prompts)

In [2]:
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM, 
    AutoTokenizer, 
    BitsAndBytesConfig, 
    TrainingArguments
)
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer

# --- 1. Configuration ---
model_name = "meta-llama/Meta-Llama-3-8B" # Ensure you have Hugging Face access to the model
# dataset_path = "train_data.jsonl"
# Instead of pointing to one file, point to the folder containing all your jsonl files
dataset_path = "./Data/json/*.jsonl"
output_dir = "./scene_safety_adapter"


# --- 2. Load and Format Dataset ---
dataset = load_dataset("json", data_files=dataset_path, split="train")
print(f"Loaded {len(dataset)} total examples from the folder!")

# def format_prompts(example):
#     """Combines instruction, input JSON, and the target output into one prompt string."""
#     prompt = f"Instruction: {example['instruction']}\nInput: {example['input']}\nOutput: {example['output']}"
#     return {"text": prompt}

dataset = dataset.map(format_prompts)

# --- 3. Load Model in 4-bit (QLoRA) ---
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

# --- 4. Setup LoRA (The fine-tuning adapter) ---
peft_config = LoraConfig(
    r=16, 
    lora_alpha=32, 
    target_modules=["q_proj", "v_proj"], # Target attention blocks
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, peft_config)

# --- 5. Training Arguments ---
training_args = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    max_steps=500, # Adjust based on how large your dataset is
    save_steps=100,
    optim="paged_adamw_8bit",
    fp16=True, 
)

# --- 6. Initialize Trainer ---
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    dataset_text_field="text",
    max_seq_length=512, # Scene graphs are text-light, so 512 tokens is plenty
    tokenizer=tokenizer,
    args=training_args,
)

# --- 7. Start Training ---
print("Starting Fine-Tuning...")
trainer.train()

# --- 8. Save the Fine-Tuned Adapter ---
trainer.model.save_pretrained(f"{output_dir}/final_adapter")
tokenizer.save_pretrained(f"{output_dir}/final_adapter")
print("Training Complete! Adapter saved.")

c:\Users\mt5864s\AppData\Local\miniconda3\envs\new_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating train split: 10000 examples [00:00, 85512.70 examples/s]


Loaded 10000 total examples from the folder!


Map: 100%|██████████| 10000/10000 [00:03<00:00, 3100.30 examples/s]


OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/meta-llama/Meta-Llama-3-8B.
401 Client Error. (Request ID: Root=1-69b8c08e-45461fdf4cbfdd17165b456d;133489a3-ddb3-4fb6-a44e-807a6671b08b)

Cannot access gated repo for url https://huggingface.co/meta-llama/Meta-Llama-3-8B/resolve/main/config.json.
Access to model meta-llama/Meta-Llama-3-8B is restricted. You must have access to it and be authenticated to access it. Please log in.

In [5]:
import torch
import json
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM, 
    AutoTokenizer, 
    BitsAndBytesConfig, 
    TrainingArguments
)
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer

# --- 1. Configuration ---
# Using an ungated, highly capable 3-billion parameter model
model_name = "Qwen/Qwen2.5-3B-Instruct" 
dataset_path = "./Data/json/*.jsonl"
output_dir = "./scene_safety_qwen_adapter"

# --- 2. Load and Format Dataset ---
dataset = load_dataset("json", data_files=dataset_path, split="train")

def format_prompts(example):
    """Isolates the target event and formats the prompt."""
    target_event = example.get("event", "Safe") # Fallback just in case
    
    input_dict = {
        "environment": example.get("environment", ""),
        "nodes": example.get("nodes", []),
        "edges": example.get("edges", [])
    }
    
    input_json_str = json.dumps(input_dict)
    
    instruction = (
        "Analyze the scene graph JSON and classify the safety event. "
        "The classes are: 'Safe', 'Gun Detection', 'Knife Detection', 'Fall Detection', 'Fire Detection'."
    )
    
    # Instruction-Input-Output format
    full_prompt = f"Instruction: {instruction}\nInput: {input_json_str}\nOutput: {target_event}"
    return {"text": full_prompt}

# Apply the formatting to the dataset
dataset = dataset.map(format_prompts)

# --- 3. Load Model in 4-bit (QLoRA) ---
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
# Qwen doesn't have a default pad token, so we map it to the EOS token
tokenizer.pad_token = tokenizer.eos_token 

print(f"Downloading and loading {model_name}...")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

# --- 4. Setup LoRA (The fine-tuning adapter) ---
peft_config = LoraConfig(
    r=16, 
    lora_alpha=32, 
    # "all-linear" automatically finds the right attention layers for ANY model architecture
    target_modules="all-linear", 
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
# model = get_peft_model(model, peft_config)

# --- 5. Training Arguments ---
training_args = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    max_steps=500, # Adjust based on dataset size (500 is a good starting point)
    save_steps=100,
    optim="paged_adamw_8bit",
    fp16=True, 
)

# --- 6. Initialize Trainer ---
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    # dataset_text_field="text",
    # max_seq_length=512, # 512 is plenty for your JSON graphs
    # tokenizer=tokenizer,
    args=training_args,
)

# --- 7. Start Training ---
print("Starting Fine-Tuning...")
trainer.train()

# --- 8. Save the Fine-Tuned Adapter ---
trainer.model.save_pretrained(f"{output_dir}/final_adapter")
tokenizer.save_pretrained(f"{output_dir}/final_adapter")
print(f"Training Complete! Adapter saved to {output_dir}/final_adapter")

Truncating train dataset: 100%|██████████| 10000/10000 [00:00<00:00, 208543.19 examples/s]
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Starting Fine-Tuning...


c:\Users\mt5864s\AppData\Local\miniconda3\envs\new_env\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


RuntimeError: All input tensors need to be on the same GPU, but found some tensors to not be on a GPU:
 [(torch.Size([16, 2048]), device(type='cpu')), (torch.Size([16, 2048]), device(type='cpu')), (torch.Size([16, 2048]), device(type='cpu')), (torch.Size([16, 2048]), device(type='cpu')), (torch.Size([256]), device(type='cpu')), (torch.Size([256]), device(type='cpu')), (torch.Size([128]), device(type='cpu')), (torch.Size([128]), device(type='cpu'))]

In [9]:
import torch
import json
import os
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM, 
    AutoTokenizer, 
    BitsAndBytesConfig, 
    TrainingArguments
)
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

# --- 0. STRICT GPU CHECK ---
# assert torch.cuda.is_available(), "CRITICAL ERROR: GPU is not detected! PyTorch is running on CPU."
# print(f"Success: Training on {torch.cuda.get_device_name(0)}")

# --- 1. Configuration ---
model_name = "Qwen/Qwen2.5-3B-Instruct" 
dataset_path = "./Data/json/*.jsonl"
output_dir = "./scene_safety_qwen_adapter"

# --- 2. Load and Format Dataset ---
dataset = load_dataset("json", data_files=dataset_path, split="train")

def format_prompts(example):
    target_event = example.get("event", "Safe") 
    input_dict = {
        "environment": example.get("environment", ""),
        "nodes": example.get("nodes", []),
        "edges": example.get("edges", [])
    }
    input_json_str = json.dumps(input_dict)
    instruction = (
        "Analyze the scene graph JSON and classify the safety event. "
        "The classes are: 'Safe', 'Gun Detection', 'Knife Detection', 'Fall Detection', 'Fire Detection'."
    )
    full_prompt = f"Instruction: {instruction}\nInput: {input_json_str}\nOutput: {target_event}"
    return {"text": full_prompt}

dataset = dataset.map(format_prompts)

# --- 3. Load Model in 4-bit ---
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token 

# OPTIMIZATION: Use Flash Attention 2 if hardware supports it
attn_implementation = "flash_attention_2" if torch.cuda.is_bf16_supported() else "sdpa"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation=attn_implementation
)

# --- 4. Setup LoRA ---
peft_config = LoraConfig(
    r=16, 
    lora_alpha=32, 
    target_modules="all-linear", 
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# # --- 5. OPTIMIZED Training Arguments ---
# training_args = TrainingArguments(
#     output_dir=output_dir,
    
#     # SPEED HACK: Double the batch size to keep the GPU fully fed
#     per_device_train_batch_size=8, 
#     # Compensate by dropping accumulation steps to maintain effective batch size
#     gradient_accumulation_steps=2, 
    
#     # SPEED HACK: Use faster precision if your GPU allows it
#     bf16=torch.cuda.is_bf16_supported(),
#     fp16=not torch.cuda.is_bf16_supported(),
    
#     # SPEED HACK: Train for 2 passes over the data instead of 500 arbitrary steps
#     num_train_epochs=2, 
    
#     # SPEED HACK: Use multiple CPU cores to load data faster
#     dataloader_num_workers=int(os.cpu_count() or 2), 
    
#     learning_rate=2e-4,
#     logging_steps=10,
#     save_strategy="epoch", # Save once per epoch instead of every 100 steps
#     optim="paged_adamw_8bit",
# )

# --- 5. OPTIMIZED Training Arguments (Using the new SFTConfig) ---
training_args = SFTConfig(
    output_dir=output_dir,
    per_device_train_batch_size=8, 
    gradient_accumulation_steps=2, 
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    num_train_epochs=2, 
    dataloader_num_workers=int(os.cpu_count() or 2), 
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="epoch", 
    optim="paged_adamw_8bit",
    
    # NEW: These parameters now live inside the config!
    max_length=256, 
    dataset_text_field="text"
)

# # --- 6. Initialize Trainer ---
# trainer = SFTTrainer(
#     model=model,
#     train_dataset=dataset,
#     peft_config=peft_config,
#     dataset_text_field="text",
#     # SPEED HACK: Keep sequence length tight so you aren't calculating empty padding
#     max_seq_length=256, 
#     tokenizer=tokenizer,
#     args=training_args,
# )

# --- 6. Initialize Trainer ---
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    processing_class=tokenizer, # NEW: 'tokenizer' was renamed to 'processing_class'
    args=training_args,
)

# --- 7. Start Training ---
print("Starting Optimized Fine-Tuning...")
trainer.train()

# --- 8. Save the Fine-Tuned Adapter ---
trainer.model.save_pretrained(f"{output_dir}/final_adapter")
tokenizer.save_pretrained(f"{output_dir}/final_adapter")
print(f"Training Complete! Adapter saved to {output_dir}/final_adapter")

Truncating train dataset: 100%|██████████| 10000/10000 [00:00<00:00, 235539.78 examples/s]
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


Starting Optimized Fine-Tuning...


c:\Users\mt5864s\AppData\Local\miniconda3\envs\new_env\Lib\site-packages\torch\utils\data\dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


RuntimeError: All input tensors need to be on the same GPU, but found some tensors to not be on a GPU:
 [(torch.Size([16, 2048]), device(type='cpu')), (torch.Size([16, 2048]), device(type='cpu')), (torch.Size([16, 2048]), device(type='cpu')), (torch.Size([16, 2048]), device(type='cpu')), (torch.Size([256]), device(type='cpu')), (torch.Size([256]), device(type='cpu')), (torch.Size([128]), device(type='cpu')), (torch.Size([128]), device(type='cpu'))]

In [12]:
import torch
import json
from datasets import load_dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    TrainingArguments, 
    Trainer,
    DataCollatorWithPadding
)

# --- 0. HARDWARE CHECK ---
# assert torch.cuda.is_available(), "GPU is not detected! PyTorch is running on CPU."
# print(f"Training on: {torch.cuda.get_device_name(0)}")

# --- 1. Configuration ---
model_name = "distilbert-base-uncased"
dataset_path = "./Data/json/*.jsonl"
output_dir = "./scene_safety_distilbert"

# --- 2. Label Mapping ---
# Encoder models require integer labels, not strings.
label2id = {
    "Safe": 0, 
    "Gun Detection": 1, 
    "Knife Detection": 2, 
    "Fall Detection": 3, 
    "Fire Detection": 4
}
id2label = {v: k for k, v in label2id.items()}

# --- 3. Load Dataset & Tokenizer ---
dataset = load_dataset("json", data_files=dataset_path, split="train")
tokenizer = AutoTokenizer.from_pretrained(model_name)

# --- 4. Data Preprocessing ---
def preprocess_function(examples):
    """Converts JSON structure to text and maps string labels to integers."""
    texts = []
    
    # Process the batch of data
    for env, nodes, edges in zip(examples.get('environment', []), examples.get('nodes', []), examples.get('edges', [])):
        input_dict = {
            "environment": env,
            "nodes": nodes,
            "edges": edges
        }
        # Convert dictionary to a flat string for the model to read
        texts.append(json.dumps(input_dict))
    
    # Tokenize the text strings
    tokenized_inputs = tokenizer(texts, truncation=True, max_length=256)
    
    # Map the "event" strings to their integer IDs
    tokenized_inputs["labels"] = [label2id.get(event, 0) for event in examples.get('event', [])]
    
    return tokenized_inputs

# Apply preprocessing (batched=True makes this incredibly fast)
print("Tokenizing dataset...")
tokenized_dataset = dataset.map(preprocess_function, batched=True, remove_columns=dataset.column_names)

# --- 5. Load the Classification Model ---
# We tell the model exactly how many labels to expect and how to name them
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)

# --- 6. Training Arguments ---
training_args = TrainingArguments(
    output_dir=output_dir,
    learning_rate=2e-5, # Standard learning rate for full fine-tuning
    per_device_train_batch_size=16, # DistilBERT is tiny, you can use a large batch size!
    num_train_epochs=3, # 3 passes over the dataset is usually perfect
    weight_decay=0.01,
    logging_steps=10,
    save_strategy="epoch",
    fp16=torch.cuda.is_available(), # Use mixed precision for speed
)

# Efficiently pads batches to the longest sequence in the batch (saves memory)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# --- 7. Initialize Trainer ---
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

# --- 8. Start Training ---
print("Starting Lightning-Fast Fine-Tuning...")
trainer.train()

# --- 9. Save the Model ---
trainer.save_model(f"{output_dir}/final_model")
print(f"Training Complete! Model saved to {output_dir}/final_model")

c:\Users\mt5864s\AppData\Local\miniconda3\envs\new_env\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\mt5864s\.cache\huggingface\hub\models--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Tokenizing dataset...


Map: 100%|██████████| 10000/10000 [05:27<00:00, 30.57 examples/s] 
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\mt5864s\AppData\Local\Temp\ipykernel_33396\2901339483.py:88: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Starting Lightning-Fast Fine-Tuning...


c:\Users\mt5864s\AppData\Local\miniconda3\envs\new_env\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
10,1.620400
20,1.551600
30,1.340000
40,1.098400
50,0.979100
60,0.838100
70,0.700900
80,0.687100
90,0.668200
100,0.526500


c:\Users\mt5864s\AppData\Local\miniconda3\envs\new_env\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
c:\Users\mt5864s\AppData\Local\miniconda3\envs\new_env\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Complete! Model saved to ./scene_safety_distilbert/final_model


In [13]:
from transformers import pipeline
import json

# 1. Load your newly trained local model
classifier = pipeline("text-classification", model="./scene_safety_distilbert/final_model", device=0)

# 2. Your raw scene graph data (mocking the output from your YOLO SGG)
sample_scene = {
    "environment": "Living Room",
    "nodes": [
        {"id": "n1", "label": "Person"}, 
        {"id": "n2", "label": "Gun"}
    ],
    "edges": [
        {"source": "n1", "target": "n2", "relationship": "touching_or_holding"}
    ]
}

# 3. Classify!
input_string = json.dumps(sample_scene)
result = classifier(input_string)

print("\n--- INFERENCE RESULT ---")
# Because we mapped id2label during training, it outputs the exact string!
print(f"Predicted Event : {result[0]['label']}")
print(f"Confidence Score: {result[0]['score']:.4f}")

Device set to use cpu



--- INFERENCE RESULT ---
Predicted Event : Gun Detection
Confidence Score: 0.9344


In [14]:
## For binary classification
import torch
import json
from datasets import load_dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    TrainingArguments, 
    Trainer,
    DataCollatorWithPadding,
    pipeline
)

# # ==========================================
# # 0. HARDWARE CHECK
# # ==========================================
# assert torch.cuda.is_available(), "GPU is not detected! PyTorch is running on CPU."
# print(f"Training on: {torch.cuda.get_device_name(0)}")

# ==========================================
# 1. CONFIGURATION & LABEL MAPPING
# ==========================================
model_name = "distilbert-base-uncased"
dataset_path = "./Data/json/*.jsonl"
output_dir = "./scene_safety_binary_model"

# Binary Classification Mapping
label2id = {
    "Safe": 0, 
    "Unsafe": 1 
}
id2label = {v: k for k, v in label2id.items()}

# ==========================================
# 2. DATASET PREPARATION
# ==========================================
print("Loading dataset and tokenizer...")
dataset = load_dataset("json", data_files=dataset_path, split="train")
tokenizer = AutoTokenizer.from_pretrained(model_name)

def preprocess_function(examples):
    """Converts JSON structure to text and forces binary labels."""
    texts = []
    
    # 1. Format the Input JSON Strings
    for env, nodes, edges in zip(examples.get('environment', []), examples.get('nodes', []), examples.get('edges', [])):
        input_dict = {
            "environment": env,
            "nodes": nodes,
            "edges": edges
        }
        texts.append(json.dumps(input_dict))
    
    # 2. Tokenize
    tokenized_inputs = tokenizer(texts, truncation=True, max_length=256)
    
    # 3. Apply Binary Logic to Labels
    binary_labels = []
    for event in examples.get('event', []):
        if event == "Safe":
            binary_labels.append(label2id["Safe"])
        else:
            # "Gun Detection", "Fire Detection", etc., all become "Unsafe"
            binary_labels.append(label2id["Unsafe"])
            
    tokenized_inputs["labels"] = binary_labels
    
    return tokenized_inputs

print("Tokenizing and mapping to binary categories...")
tokenized_dataset = dataset.map(preprocess_function, batched=True, remove_columns=dataset.column_names)

# ==========================================
# 3. MODEL INITIALIZATION
# ==========================================
print("Loading DistilBERT...")
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)

# ==========================================
# 4. TRAINING SETUP
# ==========================================
training_args = TrainingArguments(
    output_dir=output_dir,
    learning_rate=2e-5, 
    per_device_train_batch_size=16, 
    num_train_epochs=3, 
    weight_decay=0.01,
    logging_steps=10,
    save_strategy="epoch",
    fp16=torch.cuda.is_available(), 
)

# Efficiently pads batches to save memory
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

# ==========================================
# 5. EXECUTE TRAINING
# ==========================================
print("\n--- Starting Lightning-Fast Fine-Tuning ---")
trainer.train()

trainer.save_model(f"{output_dir}/final_model")
tokenizer.save_pretrained(f"{output_dir}/final_model")
print(f"\nTraining Complete! Model saved to {output_dir}/final_model")


# ==========================================
# 6. INSTANT INFERENCE TEST
# ==========================================
print("\n--- Running Inference Test ---")

# Load the freshly trained model into an easy-to-use pipeline
classifier = pipeline("text-classification", model=f"{output_dir}/final_model", tokenizer=f"{output_dir}/final_model", device=0)

# A sample JSON scene graph to test (Note the Gun)
sample_scene = {
    "environment": "Office",
    "nodes": [
        {"id": "n1", "label": "Person"}, 
        {"id": "n2", "label": "Gun"}
    ],
    "edges": [
        {"source": "n1", "target": "n2", "relationship": "touching_or_holding"}
    ]
}

input_string = json.dumps(sample_scene)
result = classifier(input_string)

print(f"Input Scene: Person holding a Gun in an Office")
print(f"Predicted Event  : {result[0]['label']}")
print(f"Confidence Score : {result[0]['score']:.4f}")

Loading dataset and tokenizer...
Tokenizing and mapping to binary categories...


Map: 100%|██████████| 10000/10000 [00:06<00:00, 1530.70 examples/s]
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\mt5864s\AppData\Local\Temp\ipykernel_33396\2156605581.py:101: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Loading DistilBERT...

--- Starting Lightning-Fast Fine-Tuning ---


c:\Users\mt5864s\AppData\Local\miniconda3\envs\new_env\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
10,0.282400
20,0.034100
30,0.012400
40,0.006900
50,0.004500
60,0.003300
70,0.002600
80,0.002100
90,0.001800
100,0.001500


c:\Users\mt5864s\AppData\Local\miniconda3\envs\new_env\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
c:\Users\mt5864s\AppData\Local\miniconda3\envs\new_env\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Device set to use cpu



Training Complete! Model saved to ./scene_safety_binary_model/final_model

--- Running Inference Test ---
Input Scene: Person holding a Gun in an Office
Predicted Event  : Unsafe
Confidence Score : 1.0000


In [15]:
## For binary classification
import torch
import json
from datasets import load_dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    TrainingArguments, 
    Trainer,
    DataCollatorWithPadding,
    pipeline
)

# # ==========================================
# # 0. HARDWARE CHECK
# # ==========================================
# assert torch.cuda.is_available(), "GPU is not detected! PyTorch is running on CPU."
# print(f"Training on: {torch.cuda.get_device_name(0)}")

# ==========================================
# 1. CONFIGURATION & LABEL MAPPING
# ==========================================
model_name = "distilbert-base-uncased"
dataset_path = "./Data/json/master_train_data.jsonl"
output_dir = "./scene_safety_binary_model_v2"

# Binary Classification Mapping
label2id = {
    "Safe": 0, 
    "Unsafe": 1 
}
id2label = {v: k for k, v in label2id.items()}

# ==========================================
# 2. DATASET PREPARATION
# ==========================================
print("Loading dataset and tokenizer...")
dataset = load_dataset("json", data_files=dataset_path, split="train")
tokenizer = AutoTokenizer.from_pretrained(model_name)

def preprocess_function(examples):
    """Converts JSON structure to text and forces binary labels."""
    texts = []
    
    # 1. Format the Input JSON Strings
    for env, nodes, edges in zip(examples.get('environment', []), examples.get('nodes', []), examples.get('edges', [])):
        input_dict = {
            "environment": env,
            "nodes": nodes,
            "edges": edges
        }
        texts.append(json.dumps(input_dict))
    
    # 2. Tokenize
    tokenized_inputs = tokenizer(texts, truncation=True, max_length=256)
    
    # 3. Apply Binary Logic to Labels
    binary_labels = []
    for event in examples.get('event', []):
        if event == "Safe":
            binary_labels.append(label2id["Safe"])
        else:
            # "Gun Detection", "Fire Detection", etc., all become "Unsafe"
            binary_labels.append(label2id["Unsafe"])
            
    tokenized_inputs["labels"] = binary_labels
    
    return tokenized_inputs

print("Tokenizing and mapping to binary categories...")
tokenized_dataset = dataset.map(preprocess_function, batched=True, remove_columns=dataset.column_names)

# ==========================================
# 3. MODEL INITIALIZATION
# ==========================================
print("Loading DistilBERT...")
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)

# ==========================================
# 4. TRAINING SETUP
# ==========================================
training_args = TrainingArguments(
    output_dir=output_dir,
    learning_rate=2e-5, 
    per_device_train_batch_size=16, 
    num_train_epochs=3, 
    weight_decay=0.01,
    logging_steps=10,
    save_strategy="epoch",
    fp16=torch.cuda.is_available(), 
)

# Efficiently pads batches to save memory
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

# ==========================================
# 5. EXECUTE TRAINING
# ==========================================
print("\n--- Starting Lightning-Fast Fine-Tuning ---")
trainer.train()

trainer.save_model(f"{output_dir}/final_model")
tokenizer.save_pretrained(f"{output_dir}/final_model")
print(f"\nTraining Complete! Model saved to {output_dir}/final_model")


# ==========================================
# 6. INSTANT INFERENCE TEST
# ==========================================
print("\n--- Running Inference Test ---")

# Load the freshly trained model into an easy-to-use pipeline
classifier = pipeline("text-classification", model=f"{output_dir}/final_model", tokenizer=f"{output_dir}/final_model", device=0)

# A sample JSON scene graph to test (Note the Gun)
sample_scene = {
    "environment": "Office",
    "nodes": [
        {"id": "n1", "label": "Person"}, 
        {"id": "n2", "label": "Gun"}
    ],
    "edges": [
        {"source": "n1", "target": "n2", "relationship": "touching_or_holding"}
    ]
}

input_string = json.dumps(sample_scene)
result = classifier(input_string)

print(f"Input Scene: Person holding a Gun in an Office")
print(f"Predicted Event  : {result[0]['label']}")
print(f"Confidence Score : {result[0]['score']:.4f}")

Loading dataset and tokenizer...


Generating train split: 2000 examples [00:00, 55505.54 examples/s]


Tokenizing and mapping to binary categories...


Map: 100%|██████████| 2000/2000 [00:00<00:00, 3047.49 examples/s]
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\mt5864s\AppData\Local\Temp\ipykernel_33396\955754732.py:101: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Loading DistilBERT...

--- Starting Lightning-Fast Fine-Tuning ---


c:\Users\mt5864s\AppData\Local\miniconda3\envs\new_env\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
10,0.688700
20,0.662500
30,0.530400
40,0.338500
50,0.276300
60,0.306900
70,0.237000
80,0.155100
90,0.285400
100,0.258300


c:\Users\mt5864s\AppData\Local\miniconda3\envs\new_env\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
c:\Users\mt5864s\AppData\Local\miniconda3\envs\new_env\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Device set to use cpu



Training Complete! Model saved to ./scene_safety_binary_model_v2/final_model

--- Running Inference Test ---
Input Scene: Person holding a Gun in an Office
Predicted Event  : Unsafe
Confidence Score : 0.9972
